# Introduction to Algebraic Modeling with Pyomo

In this session we introduce algebraic modeling of linear programs using Pyomo.
The goal is to translate mathematical formulations directly into executable models.

## 1. The Simplest LP

Maximize 2x subject to x ≤ 5, x ≥ 0.

In [1]:
import pyomo.environ as pyo
from orlab.solver import solve, display_solution

In [2]:
model = pyo.ConcreteModel()

model.x = pyo.Var(domain=pyo.NonNegativeReals)

model.obj = pyo.Objective(
    expr=2 * model.x,
    sense=pyo.maximize
)

model.c = pyo.Constraint(expr=model.x <= 5)

solve(model)
display_solution(model)


Optimal Solution
--------------------
xNone = 5.0000
--------------------
Objective = 10.0000


## 2. Indexed Linear Program

Maximize sum_i c_i x_i
Subject to resource constraints.

In [3]:
model = pyo.ConcreteModel()

model.I = pyo.Set(initialize=["x1", "x2"])
model.J = pyo.Set(initialize=["labor", "material"])

In [4]:
profit = {"x1": 3, "x2": 5}

A = {
    ("x1", "labor"): 2,
    ("x2", "labor"): 1,
    ("x1", "material"): 1,
    ("x2", "material"): 2,
}

capacity = {"labor": 6, "material": 6}

model.c = pyo.Param(model.I, initialize=profit)
model.a = pyo.Param(model.I, model.J, initialize=A)
model.b = pyo.Param(model.J, initialize=capacity)

In [5]:
model.x = pyo.Var(model.I, domain=pyo.NonNegativeReals)

In [6]:
def objective_rule(m):
    return sum(m.c[i] * m.x[i] for i in m.I)

model.obj = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

In [7]:
def capacity_rule(m, j):
    return sum(m.a[i, j] * m.x[i] for i in m.I) <= m.b[j]

model.capacity = pyo.Constraint(model.J, rule=capacity_rule)

In [8]:
solve(model)
display_solution(model)


Optimal Solution
--------------------
xx1 = 2.0000
xx2 = 2.0000
--------------------
Objective = 16.0000


## 3. Dual Values (Shadow Prices)

In [9]:
model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
solve(model)

for j in model.J:
    print(f"Dual value for {j} =", model.dual[model.capacity[j]])

Dual value for labor = 0.333333333333333
Dual value for material = 2.33333333333333
